# Model A: Workout Progression Classifier

This notebook trains Model A as an independent XGBoost classifier for next-session PR probability.

Outputs:
- archive/models/model_A/model_A_workout.joblib
- archive/models/model_A/model_A_metrics.json
- archive/models/model_A/model_A_predictions.csv
- archive/models/model_A/feature_importance_A.json

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
OUT = ROOT / 'archive' / 'models' / 'model_A'
OUT.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)

In [ ]:
def load_data() -> pd.DataFrame:
    return pd.read_csv(DATA / 'processed_merged.csv', parse_dates=['date'])


def add_training_age(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values(['exercise_title', 'date']).reset_index(drop=True)
    grp = df.groupby('exercise_title')
    df['training_age_sessions'] = grp.cumcount() + 1
    first_date = grp['date'].transform('min')
    df['training_age_days'] = (
        pd.to_datetime(df['date']).dt.normalize() - pd.to_datetime(first_date).dt.normalize()
    ).dt.days
    return df


def add_recent_training_context(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values(['exercise_title', 'date']).set_index('date')
    out_frames = []
    for _, g in df.groupby('exercise_title'):
        g = g.sort_index()
        g['days_since_last_workout'] = g.index.to_series().diff().dt.days.fillna(9999)
        out_frames.append(g.reset_index())
    return pd.concat(out_frames, ignore_index=True).sort_values(['exercise_title', 'date']).reset_index(drop=True)


def ensure_model_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if 'pr_gap_percent' not in df.columns and {'rolling_best_prev', 'best_est_1RM'}.issubset(df.columns):
        denom = df['rolling_best_prev'].replace(0, np.nan)
        df['pr_gap_percent'] = (df['rolling_best_prev'] - df['best_est_1RM']) / denom
    if 'pr_gap_percent' not in df.columns:
        df['pr_gap_percent'] = 0.0

    if 'volume_ratio_28_56' not in df.columns and {'volume_28d_avg', 'volume_56d_avg'}.issubset(df.columns):
        denom = df['volume_56d_avg'].replace(0, np.nan)
        df['volume_ratio_28_56'] = df['volume_28d_avg'] / denom
    if 'volume_ratio_28_56' not in df.columns:
        df['volume_ratio_28_56'] = 0.0

    for col in ['days_since_last_pr', 'sessions_since_last_pr', 'pr_freq_90d']:
        if col not in df.columns:
            df[col] = 0.0

    return df


def time_splits(df: pd.DataFrame, train_end: str = '2025-06-30', val_end: str = '2025-12-31'):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    train = df[df['date'] <= pd.to_datetime(train_end)].copy()
    val = df[(df['date'] > pd.to_datetime(train_end)) & (df['date'] <= pd.to_datetime(val_end))].copy()
    test = df[df['date'] > pd.to_datetime(val_end)].copy()
    return train, val, test


def select_features(df: pd.DataFrame, feat_list: list[str]) -> list[str]:
    seen = set()
    out = []
    for c in feat_list:
        if c in df.columns and c not in seen:
            out.append(c)
            seen.add(c)
    return out


def evaluate_classifier(clf, X: pd.DataFrame, y: pd.Series) -> dict:
    probs = clf.predict_proba(X)[:, 1]
    return {
        'roc_auc': float(roc_auc_score(y, probs)),
        'avg_precision': float(average_precision_score(y, probs)),
        'brier': float(brier_score_loss(y, probs)),
    }


def train_xgb(X_train: pd.DataFrame, y_train: pd.Series, X_val: pd.DataFrame, y_val: pd.Series):
    params = {
        'n_estimators': 300,
        'max_depth': 5,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'use_label_encoder': False,
        'eval_metric': 'logloss',
    }
    clf = xgb.XGBClassifier(**params)
    try:
        clf.fit(X_train, y_train, early_stopping_rounds=25, eval_set=[(X_val, y_val)], verbose=False)
    except TypeError:
        clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return clf

In [ ]:
df = load_data()
df = add_training_age(df)
df = add_recent_training_context(df)
df = ensure_model_features(df)
df = df[df['PR_next_session'].notnull()].copy()

train, val, test = time_splits(df)
print('splits:', train.shape, val.shape, test.shape)

features_A = [
    'relative_strength', 'rolling_best_prev', 'best_est_1RM', 'pr_gap_percent',
    'total_volume', 'volume_28d_avg', 'volume_56d_avg', 'volume_ratio_28_56', 'volume_28d_ratio', 'volume_56d_ratio', 'volume_28d_z', 'volume_56d_z',
    'total_sets', 'total_reps', 'avg_weight', 'max_weight',
    'days_since_last_pr', 'sessions_since_last_pr', 'pr_freq_90d',
    'training_age_sessions', 'training_age_days',
]
featsA = select_features(df, features_A)
print('Model A features used:', featsA)

XtrA = train[featsA].fillna(0)
ytrA = train['PR_next_session'].astype(int)
XvA = val[featsA].fillna(0)
yvA = val['PR_next_session'].astype(int)
XtA = test[featsA].fillna(0)
ytA = test['PR_next_session'].astype(int)

clfA = train_xgb(XtrA, ytrA, XvA, yvA)
joblib.dump(clfA, OUT / 'model_A_workout.joblib')

metrics_A = {
    'train': evaluate_classifier(clfA, XtrA, ytrA),
    'val': evaluate_classifier(clfA, XvA, yvA),
    'test': evaluate_classifier(clfA, XtA, ytA),
}
(OUT / 'model_A_metrics.json').write_text(json.dumps(metrics_A, indent=2))
(OUT / 'feature_importance_A.json').write_text(json.dumps(dict(zip(featsA, clfA.feature_importances_.tolist())), indent=2))

pred_train = train[['date', 'exercise_title', 'PR_next_session']].copy()
pred_train['model_A_probability_of_next_PR'] = clfA.predict_proba(XtrA)[:, 1]
pred_val = val[['date', 'exercise_title', 'PR_next_session']].copy()
pred_val['model_A_probability_of_next_PR'] = clfA.predict_proba(XvA)[:, 1]
pred_test = test[['date', 'exercise_title', 'PR_next_session']].copy()
pred_test['model_A_probability_of_next_PR'] = clfA.predict_proba(XtA)[:, 1]

pred_all = pd.concat([pred_train, pred_val, pred_test], ignore_index=True)
pred_all.to_csv(OUT / 'model_A_predictions.csv', index=False)

print('Model A metrics:')
print(json.dumps(metrics_A, indent=2))
print('Saved:')
print('-', OUT / 'model_A_workout.joblib')
print('-', OUT / 'model_A_metrics.json')
print('-', OUT / 'model_A_predictions.csv')
print('-', OUT / 'feature_importance_A.json')